### DATA INGESTION

In [8]:
### Document Structure

from langchain_core.documents import Document

In [9]:
doc = Document(
    page_content="This is the main page content being used to create RAG",
    metadata={
        "source":"example.txt",
        "pages":1,
        "author":"Raghav",
        "created_at": "2026-08-03"
    }
)
doc



Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Raghav', 'created_at': '2026-08-03'}, page_content='This is the main page content being used to create RAG')

In [10]:
## Create a simple txt file
import os 
os.makedirs("../data/text_files", exist_ok=True)

In [11]:
sample_tests={
    "../data/text_files/sample1.txt":"This is the first sample text file. It contains some sample text for testing purposes. The content of this file is meant to be simple and straightforward, allowing for easy parsing and analysis. This file serves as a basic example of how text files can be structured and utilized in various applications.",
}

for file_path,content in sample_tests.items():
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)

print("Sample text files created successfully.")        

Sample text files created successfully.


In [12]:
## TextLoader

from langchain_community.document_loaders import TextLoader

from langchain_community. document_loaders import TextLoader

loader=TextLoader("../data/text_files/sample1.txt", encoding="utf-8")
document=loader.load()
print(document)

C:\Users\Dhank\AppData\Local\Temp\ipykernel_4108\1923386825.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


[Document(metadata={'source': '../data/text_files/sample1.txt'}, page_content='This is the first sample text file. It contains some sample text for testing purposes. The content of this file is meant to be simple and straightforward, allowing for easy parsing and analysis. This file serves as a basic example of how text files can be structured and utilized in various applications.')]


In [13]:
### Directory Loader

from langchain_community.document_loaders import DirectoryLoader


## Load all the text files from the Directory
dir_loader =  DirectoryLoader(
    "../data/text_files",
    glob = "**/*.txt", ## Pattern to match for files.
    loader_cls = TextLoader,   ##Loader class to use.
    loader_kwargs={'encoding':'utf-8'},
    show_progress=False
)

documents = dir_loader.load() 
documents

[Document(metadata={'source': '..\\data\\text_files\\sample1.txt'}, page_content='This is the first sample text file. It contains some sample text for testing purposes. The content of this file is meant to be simple and straightforward, allowing for easy parsing and analysis. This file serves as a basic example of how text files can be structured and utilized in various applications.')]

In [14]:
### Directory Loader

from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

## Load all the text files from the Directory
dir_loader =  DirectoryLoader(
    "../data/pdf",
    glob = "**/*.pdf", ## Pattern to match for files.
    loader_cls = PyPDFLoader,   ##Loader class to use.
    
    show_progress=False
)

pdf_documents = dir_loader.load() 
pdf_documents


[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-01-23T20:59:23+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-01-23T20:59:23+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\Microservices_30_Interview_Questions_Answers.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='Microservices Architecture – 30 Interview Questions with\n Detailed Answers\n1. What is a Microservice?\nA microservice is a small, independently deployable service that focuses on a single business capability. It\ncommunicates with other services over lightweight protocols (HTTP/gRPC/messaging) and can be\ndeveloped, deployed, and scaled independently.\n2. Advantages of Microservices over Monolithic architecture\nIndependent scalability, faster deployments, technology flexibility, better fault isolation, and team autonomy.\n3. Disadvantages of Micro

In [15]:
type(pdf_documents[0])

langchain_core.documents.base.Document

### EMBEDDING AND VECTOR STORE DB

In [16]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [17]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    



    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize the embedding manager 
        
            Args:
                model_name: Model name for sentence embeddings.
        """
    
        self.model_name = model_name
        self.model = None
        self._load_model()


    def _load_model(self):
        """Load the sentence transformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error Loading Model {self.model_name}: {e}")
            raise e

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts
        
            Args:
                texts: List of strings to generate embeddings for.

            Returns:
                numpy array of embeddings with shape (len(texts), embedding_dim)                
         """
        if not self.model:
            raise ValueError("Model not loaded. ") 

        print(f"Generating embeddings for {len(texts)} texts.")
        embeddings = self.model.encode(texts, show_progress_bar = True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings 


    def get_embedding_dimension(self) -> int:
        """Get the dimension of the embeddings generated by the model
        
            Returns:
                int: Dimension of the embeddings.
        """
        if not self.model:
            raise ValueError("Model not loaded. ")
        
        return self.model.get_sentence_embedding_dimension()           


## Initialize the Embedding Manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4961.34it/s]


Model loaded successfully. Embedding dimension: 384


In [18]:
from langchain_core.documents import Document

def simple_chunk_documents(documents, chunk_size=500, chunk_overlap=50):
    chunked = []
    step = max(1, chunk_size - chunk_overlap)
    for i, doc in enumerate(documents):
        text = getattr(doc, "page_content", "") or ""
        start = 0
        idx = 0
        while start < len(text):
            chunk = text[start:start+chunk_size]
            md = dict(getattr(doc, "metadata", {}) or {})
            md["parent_doc_index"] = i
            md["chunk_index"] = idx
            chunked.append(Document(page_content=chunk, metadata=md))
            start += step
            idx += 1
        if len(text) == 0:
            md = dict(getattr(doc, "metadata", {}) or {})
            md["parent_doc_index"] = i
            md["chunk_index"] = 0
            chunked.append(Document(page_content="", metadata=md))
    return chunked

chunked_pdf_documents = simple_chunk_documents(pdf_documents, chunk_size=500, chunk_overlap=50)
print("Chunks:", len(chunked_pdf_documents))


Chunks: 22



 ### VECTOR STORE

In [19]:
class VectorStore:
    """Manages document embeddings in a chromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """Initializes the vector store

            Args:
                collection_name: Name of the chromaDB collection in the vector store.
                persist_directory: Directory to persist the vector store data.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None 
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            #Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #Get or create collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata = {"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector Store initialized. Collection: {self.collection_name}")
            print(f"Existing documments in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            return


    def add_document(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of Langchain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match the number of embeddings")

        print(f"Adding {len(documents)} documents to the vector store...")

        # Prepare Data for the vector store
        ids = []
        metadatas = []
        documents_text = []
        embeddings_texts = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            #Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            #Embedding
            try:
                embeddings_texts.append(embedding.toList())
            except Exception:
                embeddings_texts.append(list(embedding))

                
        # Add to collection
        try:
            self.collection.add(
                ids = ids,
                metadatas = metadatas,
                documents = documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            return


vectorstore = VectorStore()
vectorstore


Vector Store initialized. Collection: pdf_documents
Existing documments in collection: 22


In [20]:
chunked_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-01-23T20:59:23+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-01-23T20:59:23+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\Microservices_30_Interview_Questions_Answers.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'parent_doc_index': 0, 'chunk_index': 0}, page_content='Microservices Architecture – 30 Interview Questions with\n Detailed Answers\n1. What is a Microservice?\nA microservice is a small, independently deployable service that focuses on a single business capability. It\ncommunicates with other services over lightweight protocols (HTTP/gRPC/messaging) and can be\ndeveloped, deployed, and scaled independently.\n2. Advantages of Microservices over Monolithic architecture\nIndependent scalability, faster deployments, technology flexibility, better fault isolati'),
 Doc

In [21]:
### Convert the text to embeddings
texts = [doc.page_content for doc in chunked_pdf_documents]

## Generate embeddings

embeddings = embedding_manager.generate_embeddings(texts)

## Store in the Vector DB
vectorstore.add_document(chunked_pdf_documents, embeddings)

Generating embeddings for 22 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.11it/s]


Generated embeddings with shape: (22, 384)
Adding 22 documents to the vector store...
Successfully added 22 documents to vector store
Total documents in collection: 44


In [22]:
### REtriever Pipeline from VectorStore

In [ ]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vectorstore: VectorStore, embedding_manager: EmbeddingManager):
        """
            Initializes the retriever
            
            Args:
                vectorstore: Vector Store containing the document embeddings
                embedding_manager: Manager for generating query embeddings
        """
        self.vectorstore = vectorstore
        self.embedding_manager = embedding_manager   


    def retrieve( self, query: str, top_k: int = 5, score_threshold: float = 0.0)-> List[Dict[str, Any]]:
        """
            Retrieve relevant documents for a query

            Args:
                query: The search query
                top_k: Number of top results to return
                score_threshold: Minimum similarity score threshold

            Returns:
                List of dictionaries containing retrieved documents and metadata
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")


        #Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        #Search in Vector DB
        try:
            results = self.vectorstore.collection.query(
                query_embeddings = [query_embedding.tolist()],
                n_results = top_k
            )

            #Process Results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    #Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print(f"No documents found")
                
            return retrieved_docs 
        
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever = RAGRetriever(vectorstore, embedding_manager)

In [24]:
rag_retriever

In [25]:
rag_retriever.retrieve("What are microservices?")
## This gave the context to be given to the LLM 

Retrieving documents for query: 'What are microservices?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00, 78.21it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_bc1af330_0',
  'content': 'Microservices Architecture – 30 Interview Questions with\n Detailed Answers\n1. What is a Microservice?\nA microservice is a small, independently deployable service that focuses on a single business capability. It\ncommunicates with other services over lightweight protocols (HTTP/gRPC/messaging) and can be\ndeveloped, deployed, and scaled independently.\n2. Advantages of Microservices over Monolithic architecture\nIndependent scalability, faster deployments, technology flexibility, better fault isolati',
  'metadata': {'parent_doc_index': 0,
   'creationdate': '2026-01-23T20:59:23+00:00',
   'source': '..\\data\\pdf\\Microservices_30_Interview_Questions_Answers.pdf',
   'subject': '(unspecified)',
   'doc_index': 0,
   'producer': 'ReportLab PDF Library - www.reportlab.com',
   'creator': '(unspecified)',
   'page': 0,
   'author': '(anonymous)',
   'content_length': 500,
   'title': '(anonymous)',
   'total_pages': 3,
   'page_label': '1',
   't

### Integration VectoirDB Context pripeline with LLM output 

In [30]:
### Simple RAG pipeline using groq
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

 ### Initialize the Groq LLM (set  your Groq api key in .env)
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(groq_api_key=groq_api_key, model_name="groq/compound", temperature=0.1, max_tokens=1024)

### 2. Simple RAG function: retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):
    ## Retrieve the context
    results=retriever.retrieve(query, top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
       return "No relevant context found"

    ##Generate the response using the groq llm
    prompt = """Use the following context to answer the following question concisely.
        Context:
        {context}

        Question: {query}
        
        Answer:
    """

    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content

In [32]:
answer = rag_simple("What is a microservice", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'What is a microservice'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00, 113.58it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


A microservice is a small, independently deployable component that implements a single business capability. It runs on its own, can be developed, deployed, and scaled separately, and communicates with other services via lightweight protocols such as HTTP, gRPC, or messaging.


### Enhanced RAG Pipeline Features

In [36]:
### Enhanced RAG Pipeline features
def rag_advanced(query, retriever, llm, top_k=3, min_score=0.2, return_context=False):
    """
        RAG pipeline with extra features:
        - Returns answer, sources, confidence score and optionally full context
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found', 'sources':[], 'min_score':0.0, 'context':''}

    #Prepare Context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources=[{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300]+'...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    ## Generate Answer
    prompt = f"Use the following context to answer the question precisely. \n\n Context:{context} \n Question:{query} \n Answer:"
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context: 
        output['context'] = context
    return output

result = rag_advanced("How do microservices connect to each other?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print('Answer ', result['answer'])
print('Sources ', result['sources'])
print('Confidence ', result['confidence'])
print('Context Preview: ', result['context'][:300])

Retrieving documents for query: 'How do microservices connect to each other?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00, 113.96it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer  Microservices talk to one another through lightweight, language‑agnostic communication mechanisms.  

**Typical internal connections**

| Method | How it works | Typical use |
|--------|--------------|-------------|
| **REST/HTTP** | Simple request‑response over HTTP/HTTPS using JSON, XML, etc. | Synchronous calls, CRUD‑style operations |
| **gRPC** | Binary protocol built on HTTP/2 with protobuf contracts. | High‑performance, low‑latency synchronous calls |
| **Messaging / Event‑driven** | Asynchronous messages placed on a broker (Kafka, RabbitMQ, NATS, etc.) or published as events. | Loose coupling, eventual consistency, fire‑and‑forget workflows |

These calls are usually discovered via a **service‑registry** (e.g., Consul, Eureka, etcd) so each service can locate the current address of its peers without hard‑coding URLs.

**External exposure**

When a client outside the system needs to reach the microservice landscape, an **API Gateway** (or edge service) sits in front. The

In [39]:
## Advanced RAG pipeline: Streaming, Citations, History, Summarization

from typing import List, Dict, Any
import time 


class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  #Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict:
        #Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found"
            source = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120]+'...'
            } for doc in results]

            #Streaming answer simulation
            prompt = f""" Use the following  context to answer the question precisely.  \nContext: \n{context} \n\n Question:{question}\n Answer:"""
            if stream: 
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush= True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

            #Add Citations to the answer
            citations= [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
            answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

            #Optionally summarize the answer
            summary = None 
            if summarize and answer:
                summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
                summary_resp = self.llm.invoke([summary_prompt])
                summary = summary_resp.content

            # Store query history
            self.history.append({
                'question': question,
                'answer': answer,
                'source': sources,
                'summary': summary,
            })

            return {
                'question': question,
                'answer': answer,
                'source': sources,
                'summary': summary,
                'history': self.history
            }




In [40]:
    #Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is a microservice? and how does it contact other microservices", top_k= 3, min_score= 0.1, stream=True, summarize=True)
print("\n Final answer: ", result['answer'])
print("Summary: ", result['summary'])
print("History: ", result['history'][-1])

Retrieving documents for query: 'what is a microservice? and how does it contact other microservices'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00, 35.96it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
 Use the following  context to answer the question precisely.  
Context: 
Microservices Architecture – 30 Interview Questions with
 Detailed Answers
1. What is a Microservice?
A microservice is a small, independently deployable service that focuses on a single business capability. It
communicates with other services ov

er lightweight protocols (HTTP/gRPC/messaging) and can be
developed, deployed, and scaled independently.
2. Advantages of Microservices over Monolithic architecture
Independent scalability, faster deployments, technology flexibility, better fault isolati

Microservices Architecture – 30 Interview Questions with
 Detailed Answers
1. What is a Microservice?
A microservice is a small, independently deployable service that focuses on a single business capability. It
communicates with other services over lightweight protocols (HTTP/gRPC/messaging) and can be
developed, deployed, and scaled independently.
2. Advantages of Microservices over Monolithic architecture
Independent scalability, faster deployments, technology flexibility, better fault isolati

12. How do Microservices communicate internally and externally?
Internally via REST/gRPC/messaging, externally via API Gateway.
13. Which Microservice design patterns have you used and why?
API Gateway, Circuit Breaker, SAGA, CQRS, Service Di